# Multi-Agent Systems — 에이전트 팀 만들기 (데스크탑 Ollama판)

코스 섹션 [Multi-Agent Systems](https://huggingface.co/learn/agents-course/unit2/smolagents/multi_agent_systems)를 내 데스크탑 Ollama(`qwen3.5:9b`)로 해 봐요.

**한 문장 요약: 에이전트 하나에게 다 시키지 말고, 팀장(manager) 에이전트가 팀원(managed agent) 에이전트에게 일을 나눠 맡긴다. 팀원은 자기 일만 하고 "보고서"만 돌려준다**

| 단계 | 하는 일 | 모델 필요? |
|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | 1단계부터 |
| 2 | 원문 도구 `calculate_cargo_travel_time` 모델 없이 불러 보기 | ❌ |
| 3 | 팀원 에이전트 만들기 + **팀장이 보는 팀원 설명서** 들여다보기 | ❌ |
| 4 | 팀장 + 팀원으로 일 시키기 (인터넷 없이, 내 장소 목록에서 찾기) | ✅ |
| 5 | 기억이 **정말 나뉘어** 있나 확인 (토큰 수 비교) | ❌ |
| 6 | 지도 그리기 + 계획 세우기(`planning_interval`) + 정답 검사(`final_answer_checks`) | ✅ |
| 7 | (선택) 원문처럼 **진짜 웹 검색 팀원**으로 바꾸기 | ✅ (+ 인터넷) |
| 8 | 원문 코드와 다른 점 정리 (읽기만) | — |

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 이번엔 **새 패키지 2개**(`plotly` 지도 그리기, `pandas` 표 만들기)가 필요해요 → `git pull` 뒤 `pip install -r requirements.txt` 한 번.
❌ 표시 칸은 모델을 안 불러서 **바로** 끝나요. ✅ 칸은 팀장·팀원이 번갈아 모델을 불러서 **몇 분** 걸릴 수 있어요.

## 0단계: 준비 확인

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "markdownify", "pandas", "plotly"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, `qwen3.5:9b`와 패키지 5개 전부 `True`
- `plotly 설치됨? False` → 아직 새 패키지를 안 깔았음: Jupyter를 끄고 `pip install -r requirements.txt`
- `ModuleNotFoundError` → venv를 켜지 않고 Jupyter를 열었음
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결

지난 노트북들과 **같은 설정**이에요. 원문은 팀원에 `Qwen2.5-Coder-32B`, 팀장에 더 똑똑한 `DeepSeek-R1`을 (돈 내는 Together AI로) 따로 썼는데, 우리는 **둘 다 같은 `qwen3.5:9b`** 를 써요.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말

## 2단계: 원문 도구 — 화물기로 몇 시간 걸리나 (모델 없이)

원문 도구 그대로예요. 지구 위 두 점(위도·경도) 사이 거리를 **대권 거리**(great-circle distance: 둥근 지구 위에서 가장 짧은 길, 하버사인 공식)로 구하고 → 10% 더하고(돌아가는 길·관제) → 시속 750km로 나누고 → 이착륙 1시간을 더해요.

| 말 | 뜻 |
|---|---|
| 위도(latitude) | 적도에서 남북으로 몇 도. 북쪽 +, 남쪽 − |
| 경도(longitude) | 런던(그리니치)에서 동서로 몇 도. 동쪽 +, 서쪽 − |
| `Tuple[float, float]` | 숫자 2개 묶음 `(위도, 경도)`. 모델이 보는 설명서엔 `array`로 나와요 |

> 원문 이야기엔 "**배**(boat)로 옮기는 시간"이라고 써 놓고, 도구와 실제 질문은 "**화물기**(cargo plane)"예요. 원문이 헷갈리게 써 놓은 것뿐이라, 여기선 **화물기**로 통일해요.

In [ ]:
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


GOTHAM = (40.7128, -74.0060)  # 원문: "우리는 Gotham(= 뉴욕 좌표)에 있다"

print("시카고 → 시드니:", calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)), "시간")
print("글래스고 → Gotham:", calculate_cargo_travel_time((55.8642, -4.2518), GOTHAM), "시간")

✅ **확인할 것**: `시카고 → 시드니: 22.82 시간`, `글래스고 → Gotham: 8.6 시간`
- 글래스고 8.6시간은 원문 결과표 첫 줄(Necropolis Cemetery, Glasgow — 8.60)과 같아요. 원문 에이전트도 결국 **이 도구로 계산**한 거예요
- 이 칸은 그냥 파이썬 함수 실행이라 모델이 필요 없어요 (5번 Tools 노트북 "모델 없이 도구 직접 부르기"와 같음)

## 3단계: 팀원 에이전트 만들기 + 팀장이 보는 팀원 설명서 (모델 없이)

원문 팀원은 **웹 검색 에이전트**인데, 웹 검색은 느리고 DuckDuckGo가 자주 막아요(rate limit). 그래서 먼저 **인터넷 없이** 팀 구조만 익히려고, 6번 Retrieval 노트북처럼 **내가 만든 작은 장소 목록**을 찾아보는 팀원을 만들어요. (진짜 웹 검색 팀원은 7단계)

| 원문 | 이 노트북 (4~6단계) |
|---|---|
| `web_agent`: Google 검색 + 웹페이지 읽기 | `location_agent`: 내 장소 목록 검색 (`search_places` 도구) |
| 팀장: 계산 도구 + 지도 그리기 | 같음 |

> ⚠️ 장소 목록은 원문 결과표에 나온 곳 + 유명한 슈퍼카 공장으로 **내가 넣은 연습용 데이터**예요. 좌표는 대략값이에요.

In [ ]:
from smolagents import CodeAgent

# 연습용 장소 목록 (이름, 위도, 경도, 메모)
BATMAN_FILMING = [
    ("Glasgow Necropolis, Glasgow, UK", 55.8628, -4.2353, "The Batman (2022)"),
    ("St. George's Hall, Liverpool, UK", 53.4084, -2.9799, "The Batman (2022)"),
    ("Wollaton Hall, Nottingham, UK", 52.9480, -1.2083, "The Dark Knight Rises (2012) - Wayne Manor"),
    ("Knebworth House, Hertfordshire, UK", 51.8722, -0.2150, "Batman (1989) - Wayne Manor"),
    ("Chicago, USA", 41.8781, -87.6298, "The Dark Knight (2008)"),
    ("Wall Street, New York, USA", 40.7060, -74.0088, "The Dark Knight Rises (2012)"),
    ("Mehrangarh Fort, Jodhpur, India", 26.2978, 73.0185, "The Dark Knight Rises (2012) - the pit"),
    ("Hong Kong, China", 22.2793, 114.1628, "The Dark Knight (2008)"),
]
SUPERCAR_FACTORIES = [
    ("McLaren, Woking, UK", 51.3466, -0.5470, "McLaren Technology Centre"),
    ("Ferrari, Maranello, Italy", 44.5294, 10.8656, "Ferrari factory"),
    ("Lamborghini, Sant'Agata Bolognese, Italy", 44.6597, 11.1340, "Lamborghini factory"),
    ("Pagani, San Cesario sul Panaro, Italy", 44.5617, 11.0336, "Pagani factory"),
    ("Bugatti, Molsheim, France", 48.5390, 7.4920, "Bugatti factory"),
    ("Koenigsegg, Angelholm, Sweden", 56.2928, 12.8569, "Koenigsegg factory"),
]


@tool
def search_places(query: str) -> str:
    """
    Searches Alfred's travel notebook for places and returns their names and coordinates.
    The notebook only knows Batman filming locations and supercar factories.

    Args:
        query: What to look for, e.g. 'Batman filming locations' or 'supercar factories'.
    """
    print(f"   📒 팀원이 장소 목록 검색: {query!r}")  # 관찰용 (원문엔 없는 도구)
    q = query.lower()
    rows = []
    if any(w in q for w in ["batman", "film", "movie", "gotham"]):
        rows += BATMAN_FILMING
    if any(w in q for w in ["supercar", "factory", "factories", "manufacturer"]):
        rows += SUPERCAR_FACTORIES
    if not rows:
        return "No places found. Try 'Batman filming locations' or 'supercar factories'."
    return "\n".join(f"{name} | lat={lat} | lon={lon} | {note}" for name, lat, lon, note in rows)


# 모델 없이 도구만 먼저
print(search_places("supercar factories"))

✅ **확인할 것**: `📒` 줄 다음에 슈퍼카 공장 6줄

이제 이 도구를 가진 **팀원 에이전트**를 만들어요. 그냥 에이전트와 다른 건 딱 두 칸, **`name`과 `description`** 이에요.
- `name` = 팀장이 부를 때 쓰는 **이름** (파이썬 함수 이름처럼 써야 해서 띄어쓰기·하이픈 ❌)
- `description` = 팀장이 "이 팀원은 뭘 잘하지?"를 판단하는 **소개글**

→ 5번 Tools에서 배운 **도구 설명서 4칸**(이름·설명·입력·출력)과 똑같은 역할이에요. 팀장 눈에는 **팀원도 도구 하나**로 보여요.

In [ ]:
location_agent = CodeAgent(
    model=model,
    tools=[search_places],
    name="location_agent",
    description="Finds places (Batman filming locations, supercar factories) with their latitude and longitude.",
    verbosity_level=1,  # 원문은 0 (팀원 화면 출력 끄기). 공부용으로 조금 보이게
    max_steps=4,
)

manager = CodeAgent(
    model=model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[location_agent],      # ★ 여기에 팀원을 넣으면 팀장이 됨
    additional_authorized_imports=["pandas"],
    max_steps=8,
)

manager.visualize()

✅ **확인할 것**: 원문처럼 나무(tree) 모양 그림
- 맨 위 `CodeAgent | ollama_chat/qwen3.5:9b` = **팀장**. 도구는 `calculate_cargo_travel_time`, `final_answer`
- 아래 `🤖 Managed agents:` → `location_agent` = **팀원**. 팀원의 소개글(📝)과 팀원만의 도구(`search_places`)
- 팀장의 도구 표에는 `search_places`가 **없어요.** 팀장은 직접 못 찾고, 팀원에게 시켜야 해요

### 3-B: 팀장이 실제로 보는 글 (system prompt 일부)

팀장의 **시스템 프롬프트**(에이전트 규칙서, Unit 1 Messages 섹션)에서 팀원 부분만 잘라 봐요.

In [ ]:
sp = manager.system_prompt
start = sp.find("You can also give tasks to team members")
end = sp.find("Here are the rules you should always follow")
print(sp[start:end])

✅ **확인할 것**
- 팀원이 `def location_agent(task: str, additional_args: dict) -> str:` 처럼 **파이썬 함수**로 소개돼요. 그래서 CodeAgent 팀장은 코드에서 `location_agent(task="...")` 라고 **함수 부르듯** 일을 시켜요
  - `task` = 맡길 일을 글로 자세히 / `additional_args` = 같이 넘길 자료(표·이미지 등, 안 써도 됨)
  - 돌아오는 건 `-> str`, 즉 **글(보고서)** 하나
- 재밌는 줄: *"Since this team member is a **real human**, be as detailed and verbose as necessary"* → smolagents는 팀장에게 **"팀원은 진짜 사람이니까 자세히 설명해서 시켜라"** 라고 말해요. 실제론 에이전트인데, 그래야 팀장이 일을 대충 던지지 않거든요

### 3-C: 팀원이 받는 글 + 이름·소개글이 없으면?

In [ ]:
print("----- 팀원이 받는 일감 포장지 (task 자리에 팀장의 글이 들어감) -----")
print(location_agent.prompt_templates["managed_agent"]["task"])

print("----- 팀원이 끝나고 팀장에게 돌아가는 포장지 -----")
print(location_agent.prompt_templates["managed_agent"]["report"])

print("----- 이름·소개글 없는 팀원을 넣어 보면 -----")
try:
    CodeAgent(model=model, tools=[], managed_agents=[CodeAgent(model=model, tools=[search_places])])
except AssertionError as e:
    print("에러:", e)

✅ **확인할 것**
- 팀원은 일을 받을 때 **"너는 location_agent야, 팀장이 준 일이야"** + 보고서 양식을 같이 받아요: `### 1. 짧은 결과` / `### 2. 아주 자세한 결과` / `### 3. 덧붙일 내용`
  - "한 줄로 답하지 말고 최대한 자세히" → 팀장은 팀원의 **과정을 못 보고 이 보고서만** 보니까요
- 보고서는 `Here is the final answer from your managed agent 'location_agent': ...` 로 포장돼서 팀장에게 가요
- 마지막 줄: `All managed agents need both a name and a description!` → 이름·소개글은 **필수** (모델 안 부르고 만들 때 바로 에러)

## 4단계: 팀장 + 팀원으로 일 시키기 (모델 필요, 인터넷 없이)

원문 과제를 **작게** 줄였어요: 배트맨 촬영지 + 슈퍼카 공장을 찾고 → Gotham까지 화물기 시간을 계산 → 표(pandas DataFrame)로.
마지막 줄 *"Never try to process strings using code..."* 는 **원문 팀장 질문에 있던 말**이에요 (팀원 보고서 글을 코드로 쪼개려다 실수하지 말고, `print`해서 눈으로 읽으라는 뜻).

화면에서 볼 것: 팀장 Step 안에 `location_agent(task=...)` → 그 아래 **팀원의 New run 상자**가 따로 열림 → 팀원이 끝나면 다시 팀장 Step으로.

In [ ]:
task = """Use your team member to find Batman filming locations and supercar factories, with their coordinates.
Then calculate the cargo plane transfer time from each place to Gotham (40.7128, -74.0060) with calculate_cargo_travel_time.
Return a pandas DataFrame with columns: name, kind ('batman' or 'supercar'), lat, lon, hours.

Never try to process strings using code: when you have a string to read, just print it and you'll see it."""

result = manager.run(task)
print("\n===== 최종 답 =====")
print(result)

✅ **확인할 것** (화면의 Step들을 위에서부터)
- 팀장이 팀원을 **몇 번** 불렀나? 한 번에 "둘 다 찾아 줘" 했나, **촬영지 / 공장 따로 두 번** 나눴나? (원문 저자 실행에선 "1. 배트맨 촬영지 찾기 → 2. 슈퍼카 공장 찾기"로 나눴대요)
- 팀원 상자 안에 `📒 팀원이 장소 목록 검색` 줄이 있나? = 팀원이 **자기 도구**를 쓴 것
- 팀원 결과가 `### 1. Task outcome (short version)` 양식으로 왔나?
- 화물기 시간 계산은 **팀장이** 했나? (`calculate_cargo_travel_time`은 팀장 도구)
- 마지막 표에 장소가 몇 개? 2단계처럼 Glasgow ≈ 8.6시간인가?

잘 안 됐을 때
- 팀장이 팀원을 안 부르고 **좌표를 지어냄** → 작은 모델에서 흔해요. 질문에 "Use your team member" 를 더 세게 써 보기
- `max_steps` 도달 → 팀장 8바퀴로 모자랐음. `manager.max_steps = 12`로 늘려 다시

## 5단계: 기억이 정말 나뉘어 있나 (모델 없이)

원문이 말한 멀티 에이전트의 장점 두 가지:
1. 각 에이전트가 **자기 일에만** 집중 → 더 잘함
2. 기억(memory)이 나뉘어서 **한 번에 모델에 넣는 글(입력 토큰)이 줄어듦** → 빠르고 쌈

2번을 숫자로 봐요. **입력 토큰** = 그 Step에서 모델이 읽은 글의 양. CodeAgent는 매 Step마다 **지금까지 기억 전부**를 다시 읽어서, Step이 쌓일수록 커져요.

In [ ]:
from smolagents.memory import ActionStep


def show_memory(agent, who):
    steps = [s for s in agent.memory.steps if isinstance(s, ActionStep)]
    print(f"[{who}] Step {len(steps)}개")
    for s in steps:
        tokens = s.token_usage.input_tokens if s.token_usage else "?"
        print(f"   Step {s.step_number}: 입력 토큰 {tokens}")


show_memory(manager, "팀장")
show_memory(location_agent, "팀원 (마지막으로 불린 한 번)")

print("\n----- 팀장 기억에 들어간 팀원 결과 (관찰, 앞 600자) -----")
for s in manager.memory.steps:
    if isinstance(s, ActionStep) and s.observations and "managed agent" in s.observations:
        print(s.observations[:600])
        break

✅ **확인할 것**
- 팀장의 관찰(Observation)엔 팀원의 **보고서만** 있어요. 팀원이 몇 바퀴 돌았는지, 무슨 검색 결과를 봤는지는 팀장 기억에 **없어요**
- 팀원은 불릴 때마다 **새로** 시작해요 (`run()`의 기본 `reset=True`, 6번 노트 내용). 그래서 "마지막으로 불린 한 번"만 남아 있어요
- 팀원 일이 **웹페이지 읽기**처럼 글이 긴 일이면 차이가 확 커져요: 팀원 혼자 긴 글을 읽고 → 팀장에겐 요약만. 에이전트 하나로 다 했다면 그 긴 글이 **팀장 기억에 전부 쌓여서** 매 Step 다시 읽어요 (원문: "context window is quickly filling up")
- 단점도 있어요: 팀원을 부를 때마다 모델 호출이 **더** 생겨요(팀원 몇 바퀴 + 팀장). 이 정도로 작은 일은 에이전트 하나가 더 빠를 수도 있어요

## 6단계: 지도 그리기 + 계획 세우기 + 정답 검사 (모델 필요)

원문 팀장의 나머지 3가지 기능을 붙여요.

| 기능 | 원문 코드 | 뜻 |
|---|---|---|
| 지도 그리기 | `additional_authorized_imports=[... "plotly" ...]` | 팀장 코드에서 `plotly`(그래프·지도 라이브러리)를 `import`해도 된다고 허락 |
| 계획 세우기 | `planning_interval=5` | N Step마다 **계획 Step**을 따로 끼워 넣음: "지금까지 아는 것 / 앞으로 할 일"을 정리하고 다시 시작 |
| 정답 검사 | `final_answer_checks=[check_reasoning_and_plot]` | `final_answer`를 부르면 **검사 함수**를 먼저 돌림. 실패하면 에러를 기억에 남기고 **다시 해 봄** |

원문에서 바꾼 것 (주석에 `★`, 클라우드에서 smolagents 1.26으로 직접 확인함)
1. **`"plotly"` → `"plotly.*"`**: `"plotly"`만 쓰면 `import plotly.express`가 **막혀요** (`Import of plotly.express is not allowed`). `.*`를 붙여야 그 안쪽 모듈까지 허락돼요. `px.colors...`를 쓰면 `_plotly_utils`도 필요해서 같이 넣었어요
2. **검사 함수에 `agent=None` 칸 추가**: smolagents 1.26은 검사 함수를 `check(final_answer, memory, agent=에이전트)`로 불러요. 원문 함수엔 `agent` 칸이 없어서 **매번 TypeError → 검사가 무조건 실패** → 에이전트가 답을 못 내고 맴돌아요
3. **검사 방법**: 원문은 GPT-4o(돈 드는 OpenAI 모델)에게 지도 그림을 보여 주고 PASS/FAIL을 받아요 (**LLM이 채점**). 우리는 **파이썬 코드로** "파일 있나 / 점 6개 이상인가"만 검사해요
4. **PNG → HTML**: 원문 `write_image("saved_map.png")`는 `kaleido` 패키지 + 크롬 브라우저가 따로 필요해요. HTML 파일은 추가 설치 없이 저장되고, 더블클릭하면 브라우저에서 확대·축소되는 지도로 열려요
5. `planning_interval=5` → `3` (우리 Step 수가 적어서 계획 Step을 한 번은 보려고)

In [ ]:
import os

MAP_FILE = "saved_map.html"


def check_map(final_answer, agent_memory, agent=None):  # ★ 원문엔 agent 칸이 없음 → 1.26에서 무조건 실패
    assert os.path.exists(MAP_FILE), f"Make sure to save the map with fig.write_html('{MAP_FILE}')!"
    fig = agent.python_executor.state.get("fig")
    assert fig is not None, "Store the plotly figure in a variable named fig."
    n_points = sum(len(trace.lat) for trace in fig.data if getattr(trace, "lat", None) is not None)
    assert n_points >= 6, f"The map has only {n_points} points, you need at least 6."
    print(f"   ✅ 검사 통과: 지도 파일 있음, 점 {n_points}개")
    return True


if os.path.exists(MAP_FILE):
    os.remove(MAP_FILE)  # 예전 지도가 남아 있으면 검사가 의미 없으니 지우고 시작

map_manager = CodeAgent(
    model=model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[location_agent],
    additional_authorized_imports=["pandas", "plotly.*", "_plotly_utils.*"],  # ★ 원문: "plotly" (+ geopandas, shapely, json, numpy)
    planning_interval=3,               # ★ 원문: 5
    final_answer_checks=[check_map],   # ★ 원문: GPT-4o가 그림 채점
    max_steps=12,
)

map_task = """Find Batman filming locations and supercar factories with your team member,
calculate the cargo plane transfer time to Gotham (40.7128, -74.0060) for each one. You need at least 6 points in total.
Represent this as a spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time,
and save it to saved_map.html!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale="Magma", size_max=15, zoom=1)
fig.write_html("saved_map.html", include_plotlyjs="cdn")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it."""

map_result = map_manager.run(map_task)

✅ **확인할 것**
- 맨 처음에 **`Initial plan`**(첫 계획) 줄과 계획 글이 나왔나? 안에 `## 1. Facts survey`(아는 것 / 찾아야 할 것) → `## 2. Plan`(1, 2, 3… 할 일 목록). 그 뒤 3 Step마다 **`Updated plan`**(고친 계획)이 나와요
- `final_answer(fig)` 다음에 `✅ 검사 통과` 줄이 나왔나?
  - 대신 `Check check_map failed with error: ...` 가 나오면 = 검사 **실패** → 에이전트는 그 에러 글을 보고 **다시** 해요 (예: 파일 이름을 틀렸거나 점이 6개 미만). 이게 원문의 "정답 검사"가 하는 일
- 원문 예시엔 `size=100`이라 써 있는데, 표에 없는 숫자를 넣으면 plotly가 에러를 낼 수 있어요 → 모델이 에러를 보고 스스로 고치는지도 구경거리

지도 보기: 원문처럼 **팀장의 파이썬 작업 공간**(`python_executor.state`)에서 `fig` 변수를 꺼내요. (팀장이 코드에서 만든 변수들이 여기 남아 있어요)

In [ ]:
fig = map_manager.python_executor.state.get("fig")
if fig is None:
    print("fig가 없어요 → 6단계가 지도를 못 만들고 끝났어요. 위 Step의 에러를 보고 다시 실행")
else:
    fig.show()  # 지도 바탕(타일)은 인터넷에서 받아 와요
    print("파일로도 저장됨:", os.path.abspath(MAP_FILE) if os.path.exists(MAP_FILE) else "(없음)")

✅ **확인할 것**: 세계 지도 위에 점들, 색 = 화물기 시간 (원문 그림처럼)
- 뉴욕(Wall Street) ≈ 1시간, 영국 ≈ 8~9시간, 이탈리아 공장 ≈ 10시간, 홍콩 ≈ 20시간이면 맞아요
- 지도가 안 보이면 위에 찍힌 경로의 `saved_map.html`을 **탐색기에서 더블클릭** (브라우저로 열림)
- 원문 과제의 "같은 시간이 걸리는 슈퍼카 공장"은 → 지도에서 **같은 색**인 촬영지와 공장 (예: 영국 촬영지 ↔ McLaren 공장)

## 7단계 (선택): 원문처럼 진짜 웹 검색 팀원

이제 연습용 장소 목록 대신 **인터넷을 찾는 팀원**으로 바꿔요. 원문 `web_agent` 그대로인데 바꾼 것 (주석 `★`):
1. `GoogleSearchTool(provider="serper")` → `DuckDuckGoSearchTool()`: 원문 도구는 Serper API **키**가 필요해요. DuckDuckGo는 키가 없지만 자주 막혀요(rate limit)
2. **`VisitWebpageTool(max_output_length=4000)`**: 기본값은 웹페이지 글을 **40,000자**까지 돌려줘요. 대충 1만 토큰인데 우리 모델 입력창은 `num_ctx=8192` 토큰 → 페이지 하나 읽으면 **입력창이 넘쳐서** 앞부분(규칙서!)이 잘려 나가요. 4,000자로 줄였어요 — 원문이 말한 "context window is quickly filling up" 문제를 **직접** 겪는 자리예요
3. `max_steps=10` → `6` (오래 맴돌지 않게)

⚠️ 팀원이 웹을 몇 번씩 찾고 읽어서 **10분 넘게** 걸릴 수 있어요. 중간에 멈추려면 Jupyter 위쪽 ■(Interrupt) 버튼.

In [ ]:
from smolagents import DuckDuckGoSearchTool, VisitWebpageTool

web_agent = CodeAgent(
    model=model,
    tools=[
        DuckDuckGoSearchTool(),                        # ★ 원문: GoogleSearchTool(provider="serper") (API 키 필요)
        VisitWebpageTool(max_output_length=4000),      # ★ 원문: VisitWebpageTool() (기본 40000자)
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=1,  # 원문 0
    max_steps=6,        # ★ 원문 10
)

if os.path.exists(MAP_FILE):
    os.remove(MAP_FILE)

web_manager = CodeAgent(
    model=model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=["pandas", "plotly.*", "_plotly_utils.*"],
    planning_interval=3,
    final_answer_checks=[check_map],
    max_steps=12,
)

web_manager.visualize()
web_result = web_manager.run(map_task.replace("with your team member", "on the web with your team member"))

✅ **확인할 것**
- 팀장이 `web_agent(task=...)`에 **무슨 말로** 일을 시켰나? ("real human"이라고 들었으니 자세하게 썼나?)
- 팀원 상자 안에서 `web_search` / `visit_webpage`를 몇 번 불렀나? 찾은 장소에 **좌표**까지 있었나, 아니면 좌표는 모델이 상식으로 채웠나?
- 4~6단계(내 목록)와 비교: 장소가 더 많거나 달라졌나? 시간이 얼마나 더 걸렸나?
- DuckDuckGo `RatelimitException` / 검색 결과 없음 → 잠시 뒤 다시. 계속 막히면 이 단계는 넘어가도 돼요 (구조는 4~6단계에서 다 봤어요)

지도를 다시 보고 싶으면:

In [ ]:
fig = web_manager.python_executor.state.get("fig")
fig.show() if fig is not None else print("fig 없음 → 7단계가 지도를 못 만들었어요")

## 8단계 (읽기만): 원문 코드와 다른 점 정리

| 원문 | 이 노트북 | 왜 |
|---|---|---|
| `InferenceClientModel(..., provider="together")` 팀원 Qwen2.5-Coder-32B / 팀장 DeepSeek-R1 | 둘 다 Ollama `qwen3.5:9b` | 무료·내 PC. 원문은 "팀장은 머리를 많이 써야 하니 더 센 모델"이라는 설계 — 팀마다 **다른 모델**을 줄 수 있다는 것도 멀티 에이전트 장점 |
| `GoogleSearchTool("serper")` | 4~6단계 내 장소 목록, 7단계 `DuckDuckGoSearchTool` | API 키 없이 |
| `VisitWebpageTool()` | `max_output_length=4000` | 기본 40,000자는 `num_ctx=8192`를 넘김 |
| `additional_authorized_imports=["plotly", ...]` | `"plotly.*"`, `"_plotly_utils.*"` | 1.26에서 `"plotly"`만으론 `import plotly.express`가 막힘 (직접 확인) |
| `geopandas`, `shapely` | 안 씀 | `px.scatter_map`만으로 지도가 그려져서 필요 없음 (원문은 허락만 하고 예시에서 안 씀) |
| `check_reasoning_and_plot(final_answer, agent_memory)` + GPT-4o 채점 | `check_map(..., agent=None)` + 코드 검사 | 원문 함수 모양은 1.26에서 **무조건 실패**(직접 확인). GPT-4o는 OpenAI 키·비용 |
| `fig.write_image("saved_map.png")` (`kaleido`) | `fig.write_html("saved_map.html")` | 추가 설치(크롬) 없이 |
| 첫 예시: 에이전트 **하나**(`max_steps=20`) + `planning_interval=4` 자세한 질문 | 건너뜀 | 원문도 "기억이 금방 차서 느리고 비싸다 → 팀으로 나누자"의 출발점으로만 씀. 하고 싶으면 7단계의 `web_agent`에 `map_task`를 직접 `run` |

**언제 팀으로 나누나 (감 잡기)**
- ✅ 하위 일이 **글을 많이 읽는** 일(웹 검색·문서 검색)이라 기억이 금방 찰 때 → 팀원이 읽고 요약만 넘김
- ✅ 일마다 **다른 도구·다른 모델**이 맞을 때 (검색은 빠른 모델, 계획·종합은 센 모델)
- ❌ 일이 작고 단순할 때 → 모델 호출만 늘어 더 느려요. 작은 모델일수록 "팀원에게 일 설명하기"를 잘 못할 수도

**끝!** 결과(특히 4단계 팀장이 팀원을 **몇 번, 무슨 말로** 불렀는지, 5단계 토큰 숫자, 6단계 `Initial plan`·`Updated plan`과 검사 통과/실패 줄)를 복사해서 AI에게 붙여 주면 같이 분석해요.
Jupyter 끄기: `Esc` → `Ctrl + C` 두 번.